# Data Processing Notebook for Face Mask Predictors

This notebook combines the functionality of the following scripts into one workflow:

- 00_mask_mandates.py
- 0_missing_table.py
- 1_clean_dataset.py
- 2_data_preprocessing.py
- 3_split_data.py

Expected folder structure:

Project/
├── Code/
│   └── data_processing.ipynb
├── Raw data/
│   ├── australia.csv
│   └── OxCGRT_AUS_latest.csv
└── Data/

All output files will be saved to the `Data` folder.


## Imports and paths

In [1]:
import pandas as pd
from pathlib import Path
from datetime import datetime
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split


## Set up paths

In [2]:
BASE_DIR = Path.cwd().parent
RAW_DIR = BASE_DIR / "Raw data"
DATA_DIR = BASE_DIR / "Data"

# Create the output folder if it does not already exist
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Define input file paths
AUSTRALIA_FILE = RAW_DIR / "australia.csv"
OXCGRT_FILE = RAW_DIR / "OxCGRT_AUS_latest.csv"

## 1 Process policy data and identify mandate start dates


In [3]:
# Read the OxCGRT Australia policy data
policy_df = pd.read_csv(OXCGRT_FILE)

# Keep only the columns needed for mask mandate processing
col_subsets = ["RegionName", "RegionCode", "Date", "H6M_Facial Coverings"]

# Convert the Date column to datetime and use it as the index
policy_df.index = pd.to_datetime(policy_df["Date"], format="%Y%m%d")
policy_df = policy_df.loc[:, col_subsets]

# Compute 14-day rolling averages for facial covering policies within each region
rolling_days = 14
policy_rolling = (
    policy_df.loc[:, ["RegionName", "H6M_Facial Coverings"]]
    .groupby("RegionName")
    .rolling(window=rolling_days)
    .mean()
)

# Identify the first date where the rolling average reaches the mandate threshold
mandate_limit = 3
mandate_start_df = (
    policy_rolling[policy_rolling["H6M_Facial Coverings"] >= mandate_limit]
    .groupby("RegionName")
    .head(1)
)

# Save the mandate start dates
mandate_start_file = DATA_DIR / "mandate_start_dates.csv"
mandate_start_df.to_csv(mandate_start_file)

print("Saved:", mandate_start_file)
mandate_start_df.head()


Saved: d:\Data Science Project A\All\Data\mandate_start_dates.csv


,,H6M_Facial Coverings
RegionName,Date,
Australian Capital Territory,2021-08-18,3.000000
New South Wales,2021-07-09,3.000000
Northern Territory,2021-11-21,3.000000
Queensland,2021-01-17,3.142857
South Australia,2021-07-26,3.000000


## 2 Count missing values in the Australia survey data


In [4]:
# Read the Australia survey data and treat blank strings and "__NA__" as missing values
missing_df = pd.read_csv(
    AUSTRALIA_FILE,
    na_values=[" ", "__NA__"],
    keep_default_na=True,
    low_memory=False
)

# Count missing values for every column
missing_value_counts = {}
for col in missing_df.columns:
    missing_count = missing_df[col].isna().sum()
    missing_value_counts[col] = missing_count

# Convert the results to a DataFrame and sort by missing count and variable name
missing_value_table = pd.DataFrame(
    list(missing_value_counts.items()),
    columns=["Variable Name", "Missing Value Count"]
).sort_values(by=["Missing Value Count", "Variable Name"])

# Save the missing value table
missing_value_file = DATA_DIR / "missing_value_counts.csv"
missing_value_table.to_csv(missing_value_file, index=False)

print("Saved:", missing_value_file)
missing_value_table.head()


Saved: d:\Data Science Project A\All\Data\missing_value_counts.csv


,Variable Name,Missing Value Count
0,RecordNo,0
72,age,0
76,employment_status,0
1,endtime,0
73,gender,0


## 3 Clean the Australia survey dataset

### 3.1 Helper functions for cleaning

In [5]:
def convert_datetime(dt):
    # Extract the date part and convert it to datetime
    date = dt.split()[0]
    return datetime.strptime(date, "%d/%m/%Y")


def household_convert(size_str):
    # Convert household size categories into numeric values
    for i in range(1, 8):
        if size_str == str(i):
            return i
        elif size_str == "8 or more":
            return 8
        elif size_str == "Prefer not to say" or size_str == "Don't know":
            return None


### 3.2 Load and clean the survey data

In [6]:
# Read the Australia survey data
clean_df = pd.read_csv(
    AUSTRALIA_FILE,
    na_values=[" ", "__NA__"],
    keep_default_na=True
)

# Convert the endtime column to datetime
clean_df["endtime"] = clean_df["endtime"].apply(convert_datetime)

# Drop columns with missing counts above the selected threshold
thresh_value = 10781
missing_value_table = pd.read_csv(DATA_DIR / "missing_value_counts.csv")

columns_to_drop = missing_value_table.loc[
    missing_value_table["Missing Value Count"] > thresh_value,
    "Variable Name"
].tolist()

clean_df.drop(columns=columns_to_drop, inplace=True)

print("Number of dropped columns:", len(columns_to_drop))
print("Shape after dropping sparse columns:", clean_df.shape)


C:\Users\22737\AppData\Local\Temp\ipykernel_25308\3634618615.py:2: DtypeWarning: Columns (0: i3_health, 1: i4_health, 2: i5_health_1, 3: i5_health_2, 4: i5_health_3, 5: i5_health_4, 6: i5_health_5, 7: i5_health_99, 8: i5a_health, 9: i6_health, 10: i7b_health, 11: i8_health, 12: i10_health, 13: i12_health_2, 14: i12_health_11, 15: i12_health_17, 16: i12_health_18, 17: i12_health_19, 18: i12_health_20, 19: d1_health_1, 20: d1_health_2, 21: d1_health_3, 22: d1_health_4, 23: d1_health_5, 24: d1_health_6, 25: d1_health_7, 26: d1_health_8, 27: d1_health_9, 28: d1_health_10, 29: d1_health_11, 30: d1_health_12, 31: d1_health_13, 32: d1_health_98, 33: d1_health_99, 34: WCRex2, 35: WCRV_4, 36: CORE_B2_4, 37: PHQ4_1, 38: PHQ4_2, 39: PHQ4_3, 40: PHQ4_4, 41: m1_1, 42: m1_2, 43: m1_3, 44: m1_4, 45: m4_1, 46: m4_2, 47: m4_3, 48: m4_4, 49: m4_96, 50: m4_99, 51: m4_other, 52: m5_1, 53: m5_2, 54: m6_1, 55: m6_2, 56: m6_3, 57: m6_4, 58: m6_5, 59: m6_6, 60: m6_7, 61: m6_8, 62: m6_96, 63: m6_other, 64: m7_

Number of dropped columns: 460
Shape after dropping sparse columns: (53833, 53)


### 3.3 Fill missing medical variables during the consent period

In [7]:
# Identify the date range where medical consent was not provided
sdate = pd.Timestamp("2021-02-10")
edate = pd.Timestamp("2021-10-18")
date_mask = (clean_df["endtime"] <= edate) & (clean_df["endtime"] >= sdate)

# Fill PHQ4 variables with "N/A" during the affected period
for i in range(1, 5):
    clean_df.loc[date_mask, f"PHQ4_{i}"] = clean_df.loc[date_mask, f"PHQ4_{i}"].fillna("N/A")

# Fill d1_health variables with "N/A" during the affected period
for i in range(1, 14):
    clean_df.loc[date_mask, f"d1_health_{i}"] = clean_df.loc[date_mask, f"d1_health_{i}"].fillna("N/A")

for i in range(98, 100):
    clean_df.loc[date_mask, f"d1_health_{i}"] = clean_df.loc[date_mask, f"d1_health_{i}"].fillna("N/A")


### 3.4 Drop remaining missing rows and recode variables

In [8]:
# Remove all remaining rows with missing values
clean_df.dropna(inplace=True)

# Convert perception variables from text labels into numeric scores
for i in range(1, 3):
    clean_df[f"r1_{i}"] = clean_df[f"r1_{i}"].replace({
        "7 - Agree": 7,
        "6": 6,
        "5": 5,
        "4": 4,
        "3": 3,
        "2": 2,
        "1 – Disagree": 1
    })

# Convert protective behaviour frequency responses into numeric scores
frequency_dict = {
    "Always": 5,
    "Frequently": 4,
    "Sometimes": 3,
    "Rarely": 2,
    "Not at all": 1
}

for column in clean_df.columns:
    if column.startswith("i12_health_"):
        clean_df[column] = clean_df[column].map(frequency_dict)

print("Shape after dropping missing rows:", clean_df.shape)


Shape after dropping missing rows: (41125, 53)


### 3.5 Create behavioural scales and binary targets

In [9]:
# Create the face mask behaviour scale based on the selected mask-related items
face_mask_cols = ["i12_health_1", "i12_health_22", "i12_health_23", "i12_health_25"]

clean_df["face_mask_behaviour_scale"] = clean_df[face_mask_cols].median(axis=1)
clean_df["face_mask_behaviour_binary"] = clean_df["face_mask_behaviour_scale"].apply(
    lambda x: "Yes" if x >= 4 else "No"
)

# Create the overall protective behaviour scale
protective_behaviour_cols = [col for col in clean_df.columns if col.startswith("i12_")]

clean_df["protective_behaviour_scale"] = clean_df[protective_behaviour_cols].median(axis=1)
clean_df["protective_behaviour_binary"] = clean_df["protective_behaviour_scale"].apply(
    lambda x: "Yes" if x >= 4 else "No"
)

# Create the protective behaviour scale excluding face mask items
protective_behaviour_nomask_cols = [
    col for col in protective_behaviour_cols
    if col not in face_mask_cols
]

clean_df["protective_behaviour_nomask_scale"] = clean_df[protective_behaviour_nomask_cols].median(axis=1)

clean_df[[
    "face_mask_behaviour_scale",
    "face_mask_behaviour_binary",
    "protective_behaviour_scale",
    "protective_behaviour_binary",
    "protective_behaviour_nomask_scale"
]].head()


,face_mask_behaviour_scale,face_mask_behaviour_binary,protective_behaviour_scale,protective_behaviour_binary,protective_behaviour_nomask_scale
9023,3.0,No,3.0,No,3.0
9024,4.0,Yes,3.0,No,3.0
9025,1.0,No,4.0,Yes,5.0
9026,1.0,No,1.0,No,1.0
9027,1.0,No,1.0,No,1.0


### 3.6 Collapse comorbidity variables and create week number

In [10]:
# Combine comorbidity variables into a single summary variable
d1_cols = [col for col in clean_df.columns if col.startswith("d1_")]

clean_df["d1_comorbidities"] = "Yes"
clean_df.loc[clean_df["d1_health_99"] == "Yes", "d1_comorbidities"] = "No"
clean_df.loc[clean_df["d1_health_99"] == "N/A", "d1_comorbidities"] = "NA"
clean_df.loc[clean_df["d1_health_98"] == "Yes", "d1_comorbidities"] = "Prefer_not_to_say"

# Drop the original d1 columns
clean_df = clean_df.drop(d1_cols, axis=1)

# Create a fortnight-based week number variable
start_date = clean_df["endtime"].min()
clean_df["week_number"] = ((clean_df["endtime"] - start_date).dt.days // 14) + 1


### 3.7 Convert household size, drop extra columns, save cleaned data

In [11]:
# Convert household size into numeric form
clean_df["household_size"] = clean_df["household_size"].apply(household_convert)

# Drop rows with induced missing values after household conversion
clean_df.dropna(inplace=True)

# Drop survey weighting, original qweek, and the original protective behaviour item columns
clean_df = clean_df.drop(["qweek", "weight"] + protective_behaviour_cols, axis=1)

# Save the cleaned dataset
cleaned_data_file = DATA_DIR / "cleaned_data.csv"
clean_df.to_csv(cleaned_data_file, index=False)

print("Saved:", cleaned_data_file)
print("Final cleaned dataset shape:", clean_df.shape)
clean_df.head()


Saved: d:\Data Science Project A\All\Data\cleaned_data.csv
Final cleaned dataset shape: (40136, 26)


,RecordNo,endtime,i2_health,i9_health,i11_health,age,gender,state,household_size,employment_status,...,WCRex1,r1_1,r1_2,face_mask_behaviour_scale,face_mask_behaviour_binary,protective_behaviour_scale,protective_behaviour_binary,protective_behaviour_nomask_scale,d1_comorbidities,week_number
9023,9023,2020-06-24,0.0,Not sure,Not sure,31,Male,Western Australia,1.0,Full time employment,...,Somewhat badly,5,1,3.0,No,3.0,No,3.0,Yes,1
9024,9024,2020-06-24,2.0,No,Very willing,36,Male,Victoria,4.0,Full time employment,...,Somewhat badly,6,4,4.0,Yes,3.0,No,3.0,No,1
9025,9025,2020-06-24,6.0,Yes,Very willing,73,Male,Northern Territory,2.0,Retired,...,Very well,5,6,1.0,No,4.0,Yes,5.0,Yes,1
9026,9026,2020-06-24,20.0,Yes,Somewhat willing,58,Male,Queensland,2.0,Not working,...,Somewhat badly,1,4,1.0,No,1.0,No,1.0,Yes,1
9027,9027,2020-06-24,0.0,Yes,Very willing,65,Male,Victoria,1.0,Full time employment,...,Very well,3,1,1.0,No,1.0,No,1.0,No,1


## 4 Preprocess the cleaned data for modelling


### 4.1 Helper function for mandate period assignment

In [12]:
def mandates_convert(row):
    # Convert a row into a binary indicator for whether it falls within the mandate period
    endtime = pd.to_datetime(row["endtime"], format="%Y-%m-%d")
    state = row["state"]

    if states_date[state][0] <= endtime:
        return 1
    else:
        return 0


### 4.2 Load cleaned data and mandate dates

In [13]:
# Read the cleaned survey data
preprocess_df = pd.read_csv(DATA_DIR / "cleaned_data.csv", keep_default_na=False)

# Read the derived mandate start dates
mandate_df = pd.read_csv(DATA_DIR / "mandate_start_dates.csv")

# Build a dictionary of mandate start dates by state
states_date = {}
for state, date in zip(mandate_df["RegionName"], mandate_df["Date"]):
    states_date.update({state: [date]})

for state, date_range in states_date.items():
    states_date[state] = [pd.to_datetime(date, format="%Y-%m-%d") for date in date_range]

# Create a binary mandate-period indicator
preprocess_df["within_mandate_period"] = preprocess_df.apply(mandates_convert, axis=1)

preprocess_df[["state", "endtime", "within_mandate_period"]].head()


,state,endtime,within_mandate_period
0,Western Australia,2020-06-24,0
1,Victoria,2020-06-24,0
2,Northern Territory,2020-06-24,0
3,Queensland,2020-06-24,0
4,Victoria,2020-06-24,0


### 4.3 Create dummy variables and save preprocessed data

In [14]:
# Convert selected categorical variables into dummy variables
convert_into_dummy_cols = [
    "state",
    "gender",
    "i9_health",
    "employment_status",
    "i11_health",
    "WCRex1",
    "WCRex2",
    "PHQ4_1",
    "PHQ4_2",
    "PHQ4_3",
    "PHQ4_4",
    "d1_comorbidities"
]

for col in convert_into_dummy_cols:
    dummy = pd.get_dummies(preprocess_df[col], prefix=col, drop_first=True)
    preprocess_df = pd.concat([preprocess_df, dummy], axis=1)
    preprocess_df = preprocess_df.drop(col, axis=1)

# Save the preprocessed dataset
preprocessed_file = DATA_DIR / "cleaned_data_preprocessing.csv"
preprocess_df.to_csv(preprocessed_file, index=False)

print("Saved:", preprocessed_file)
print("Preprocessed dataset shape:", preprocess_df.shape)
preprocess_df.head()


Saved: d:\Data Science Project A\All\Data\cleaned_data_preprocessing.csv
Preprocessed dataset shape: (40136, 65)


,RecordNo,endtime,i2_health,age,household_size,cantril_ladder,r1_1,r1_2,face_mask_behaviour_scale,face_mask_behaviour_binary,...,PHQ4_3_Prefer not to say,PHQ4_3_Several days,PHQ4_4_N/A,PHQ4_4_Nearly every day,PHQ4_4_Not at all,PHQ4_4_Prefer not to say,PHQ4_4_Several days,d1_comorbidities_No,d1_comorbidities_Prefer_not_to_say,d1_comorbidities_Yes
0,9023,2020-06-24,0.0,31,1.0,8.0,5,1,3.0,No,...,False,True,False,False,False,False,False,False,False,True
1,9024,2020-06-24,2.0,36,4.0,8.0,6,4,4.0,Yes,...,False,False,False,True,False,False,False,True,False,False
2,9025,2020-06-24,6.0,73,2.0,8.0,5,6,1.0,No,...,False,True,False,False,True,False,False,False,False,True
3,9026,2020-06-24,20.0,58,2.0,8.0,1,4,1.0,No,...,False,True,False,False,True,False,False,False,False,True
4,9027,2020-06-24,0.0,65,1.0,7.0,3,1,1.0,No,...,False,False,False,False,True,False,False,True,False,False
